# Week 19 Optional - Advanced MLOps Patterns

This is an OPTIONAL, async deep-dive notebook for students who finished the main Week 19 notebook (`week_19_mlops_versioning_experiments.ipynb`) and want to push their MLflow and SageMaker Model Registry skills further.

## What this notebook covers

The main notebook taught the happy path: track an experiment, run a Training Job, register a model, deploy an endpoint, and wire it back into the Week 18 supervisor. This optional notebook takes three topics the main notebook only touched on or skipped entirely, and goes deep on each.

- **Topic 1: pyfunc custom flavor.** Log a LangChain-style or Strands agent wrapper as a generic `pyfunc` so any MLflow-compatible runtime can serve it.
- **Topic 2: Nested runs and search_runs.** Run a real hyperparameter sweep with parent/child runs, then pick the winner programmatically with `mlflow.search_runs()`.
- **Topic 3: Model lineage.** Trace a deployed model back to the exact training run, data version, and git commit that produced it - i.e. answer "where did this model come from?" in production.

## Prerequisites

- Completed the main Week 19 notebook (same secret scope, same S3 bucket).
- Same Databricks cluster as the main notebook (Runtime 15.4 LTS ML).
- A registered model package from the main notebook (you will look up its ARN in Topic 3).

## Expected runtime

About 45 minutes async. Skip any topic that does not interest you. Each topic is self-contained after the setup cells (Cells 1-4).


## Environment setup

**Platform**: Azure Databricks (Runtime 15.4 LTS ML), same cluster as the main Week 19 notebook.

**Secret scope**: `aws-course-creds` (the instructor wired this up; you do not enter keys by hand).

**Shared S3 bucket**: `bread-academy-week19-shared` (same bucket as the main notebook).

**Additional cluster libraries beyond the main notebook**: none. The main notebook already installed `mlflow`, `sagemaker-mlflow`, `boto3`, `sagemaker`, and `strands-agents`. The next cell reinstalls them at the same pins, in case the cluster was restarted between notebooks.

**Auth**: every AWS call uses temporary credentials read from `dbutils.secrets`. No `getpass`, no `get_execution_role()`.


In [ ]:
%pip install --quiet "mlflow>=2.13" "sagemaker-mlflow>=0.1.0" "boto3>=1.35" "sagemaker==2.257.3" "strands-agents>=1.37,<2"
dbutils.library.restartPython()

import os
import json
import time
import uuid
from datetime import datetime
from importlib.metadata import version

import boto3
import pandas as pd
import mlflow

# Version check: use importlib.metadata, NOT pkg.__version__
for pkg in ["mlflow", "sagemaker-mlflow", "boto3", "sagemaker", "strands-agents"]:
    try:
        print(f"{pkg:25s} {version(pkg)}")
    except Exception as e:
        print(f"{pkg:25s} NOT INSTALLED ({e})")


In [ ]:
# Pull temporary AWS credentials from the Databricks secret scope. The instructor
# rotated these in before class. No getpass, no IAM role assumption from inside
# the cluster - the secret scope already holds a valid STS session.

AWS_ACCESS_KEY_ID     = dbutils.secrets.get(scope="aws-course-creds", key="aws-access-key-id")
AWS_SECRET_ACCESS_KEY = dbutils.secrets.get(scope="aws-course-creds", key="aws-secret-access-key")
AWS_SESSION_TOKEN     = dbutils.secrets.get(scope="aws-course-creds", key="aws-session-token")
SAGEMAKER_ROLE_ARN    = dbutils.secrets.get(scope="aws-course-creds", key="sagemaker-execution-role-arn")
MLFLOW_TRACKING_ARN   = dbutils.secrets.get(scope="aws-course-creds", key="mlflow-tracking-server-arn")

# Export to env so boto3 and sagemaker SDK pick them up automatically
os.environ["AWS_ACCESS_KEY_ID"]     = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_SESSION_TOKEN"]     = AWS_SESSION_TOKEN
os.environ["AWS_REGION"]            = "us-east-1"
os.environ["AWS_DEFAULT_REGION"]    = "us-east-1"

# Boto3 clients
s3_client        = boto3.client("s3",        region_name="us-east-1")
sagemaker_client = boto3.client("sagemaker", region_name="us-east-1")
sts_client       = boto3.client("sts",       region_name="us-east-1")

caller = sts_client.get_caller_identity()
print(f"Caller account: {caller['Account']}")
print(f"Caller arn:     {caller['Arn']}")

# Shared S3 bucket from the main notebook + a per-student prefix so runs do not collide
S3_BUCKET = "bread-academy-week19-shared"
USER_ID   = caller["UserId"][:8]
S3_PREFIX = f"students/{USER_ID}/optional"
EXPERIMENT_NAME = f"week19-optional-{USER_ID}"

print(f"\nS3 prefix:       s3://{S3_BUCKET}/{S3_PREFIX}")
print(f"Experiment name: {EXPERIMENT_NAME}")


In [ ]:
# Pre-flight probes: fail loud BEFORE any topic-specific code runs.
# If either of these fails, ask your instructor for help instead of plowing on.

mlflow.set_tracking_uri(MLFLOW_TRACKING_ARN)

# Probe 1: MLflow managed tracking server reachable + SigV4 auth works
try:
    experiments = mlflow.search_experiments(max_results=1)
    print(f"MLflow OK: tracking_uri={mlflow.get_tracking_uri()}")
except Exception as e:
    print(f"MLflow FAIL: {e}")
    print("Ask your instructor to confirm the SageMaker MLflow tracking server is running.")
    raise

# Probe 2: S3 bucket access
try:
    s3_client.head_bucket(Bucket=S3_BUCKET)
    print(f"S3 OK: s3://{S3_BUCKET}")
except Exception as e:
    print(f"S3 FAIL: {e}")
    raise

# Set or create the experiment for THIS notebook so runs do not leak into the main notebook's experiment
mlflow.set_experiment(EXPERIMENT_NAME)
exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
EXP_ID = exp.experiment_id
print(f"\nExperiment: {EXPERIMENT_NAME}")
print(f"Experiment ID: {EXP_ID}")


## Topic 1: Custom pyfunc flavor

MLflow ships built-in "flavors" for common frameworks: `mlflow.sklearn`, `mlflow.pytorch`, `mlflow.huggingface`, `mlflow.langchain`, and so on. The flavor handles serialization, the dependency snapshot (`conda.yaml` / `requirements.txt`), and the serving contract.

But what happens when you build something MLflow does NOT have a flavor for? A custom preprocessing pipeline. A multi-step chain combining two models with business rules. A Strands agent wrapper. For those cases there is one universal flavor: `pyfunc`.

### The contract

Subclass `mlflow.pyfunc.PythonModel` and implement two methods:

- `predict(self, context, model_input, params=None)` - required. Receives a DataFrame, numpy array, or dict; returns the prediction.
- `load_context(self, context)` - optional. Runs ONCE when the artifact is loaded back. Use it for heavy lazy initialization: building a boto3 client, downloading a model from S3, building a Strands agent. Anything that is NOT pickle-safe belongs here, not in `__init__`.

Log it with:

```python
mlflow.pyfunc.log_model(
    artifact_path="model",
    python_model=MyModel(),
    input_example=example_df,  # MLflow infers the signature from this
)
```

Source: `https://mlflow.org/docs/latest/ml/model/python_model/`

### Why this matters for Bread Financial

The fraud team has a custom decision rule that combines a DistilBERT fraud score with a business-rules table (dollar caps, geography, merchant category). If you wrap that as a pyfunc, ops can serve it behind ANY MLflow-compatible runtime - Databricks Model Serving, SageMaker, even a local Docker container - without rewriting the inference code. The pyfunc IS the serving contract.


In [ ]:
# DEMO: minimal pyfunc that wraps a business-rules decision around a fraud score.

import mlflow.pyfunc


class ThresholdAlertModel(mlflow.pyfunc.PythonModel):
    """Wraps a fraud score with a business-rules threshold and dollar cap.

    Inputs: pandas DataFrame with columns 'fraud_score' (float in 0..1) and
            'amount_usd' (float).
    Output: pandas DataFrame with columns 'action' (str) and 'reason' (str).
    """

    def __init__(self, score_threshold=0.7, amount_cap_usd=5000):
        # Plain params are pickle-safe, so __init__ is fine here.
        self.score_threshold = score_threshold
        self.amount_cap_usd  = amount_cap_usd

    def predict(self, context, model_input, params=None):
        out = []
        for _, row in model_input.iterrows():
            if row["fraud_score"] >= self.score_threshold and row["amount_usd"] >= self.amount_cap_usd:
                out.append({"action": "block",   "reason": "high_score_high_amount"})
            elif row["fraud_score"] >= self.score_threshold:
                out.append({"action": "review",  "reason": "high_score"})
            else:
                out.append({"action": "approve", "reason": "low_score"})
        return pd.DataFrame(out)


example_input = pd.DataFrame([
    {"fraud_score": 0.95, "amount_usd": 8000.0},
    {"fraud_score": 0.85, "amount_usd": 100.0},
    {"fraud_score": 0.20, "amount_usd": 2500.0},
])

with mlflow.start_run(run_name="topic1-demo-threshold-alert") as run:
    mlflow.log_params({"score_threshold": 0.7, "amount_cap_usd": 5000})
    info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=ThresholdAlertModel(score_threshold=0.7, amount_cap_usd=5000),
        input_example=example_input,
    )
    print(f"Logged model URI: {info.model_uri}")
    DEMO_MODEL_URI = info.model_uri

# Load it back and run it to prove the round-trip works
loaded = mlflow.pyfunc.load_model(DEMO_MODEL_URI)
print(loaded.predict(example_input))


### Lab 1: Wrap a Strands agent as a pyfunc

Build a `StrandsFraudExplainer` PythonModel that uses an LLM to explain whether each transaction looks like fraud.

**Steps**:

1. Take a single-column DataFrame with column `transaction_description`.
2. In `load_context()`, build a Strands `Agent` with model `us.anthropic.claude-3-haiku-20240307-v1:0` and a system prompt asking the agent to explain in one sentence whether the transaction looks like fraud. Store the agent on `self._agent`.
3. In `predict()`, iterate rows and call `self._agent(text)` for each. Return a DataFrame with two columns: `transaction_description` and `explanation`.
4. Log the pyfunc with `mlflow.pyfunc.log_model()` using the example DataFrame as `input_example`. Save the resulting `info.model_uri` into `LAB1_MODEL_URI`.
5. Reload it with `mlflow.pyfunc.load_model(LAB1_MODEL_URI)` and call `loaded.predict(example_df)`.

**Example transactions** (provided in the starter):

- `"Grocery purchase at Whole Foods - 84.32 USD"`
- `"ATM withdrawal in Lagos Nigeria - 1500 USD - card present"`
- `"Subscription renewal Netflix - 15.99 USD"`

**Critical**: store the agent on `self._agent` in `load_context()`, NOT in `__init__()`. Why? The boto3 client inside the agent is NOT pickle-safe. If you put it in `__init__`, MLflow will fail to serialize the model. Putting it in `load_context()` means the agent is rebuilt fresh every time the artifact is loaded back.

**Success criterion**: `loaded.predict(example_df)` returns three rows with non-empty `explanation` strings.


In [ ]:
# SOLUTION: Lab 1 - Wrap a Strands agent as a pyfunc

from strands import Agent


class StrandsFraudExplainer(mlflow.pyfunc.PythonModel):
    """Wrap a Strands agent so it can be served as an MLflow pyfunc."""

    def __init__(self, model_id="us.anthropic.claude-3-haiku-20240307-v1:0", system_prompt=None):
        # Plain pickle-safe attributes only.
        self.model_id = model_id
        self.system_prompt = system_prompt or (
            "You are a fraud analyst. In one sentence, explain whether the "
            "following transaction looks suspicious and why."
        )

    def load_context(self, context):
        # Build the agent here, NOT in __init__, because the agent holds a
        # boto3 client that is not pickle-safe. load_context runs AFTER the
        # artifact is loaded back, so we always get a fresh agent.
        self._agent = Agent(model=self.model_id, system_prompt=self.system_prompt)

    def predict(self, context, model_input, params=None):
        rows = []
        for _, row in model_input.iterrows():
            text = row["transaction_description"]
            msg = self._agent(text)
            rows.append({"transaction_description": text, "explanation": str(msg)})
        return pd.DataFrame(rows)


example_df = pd.DataFrame({
    "transaction_description": [
        "Grocery purchase at Whole Foods - 84.32 USD",
        "ATM withdrawal in Lagos Nigeria - 1500 USD - card present",
        "Subscription renewal Netflix - 15.99 USD",
    ]
})

LAB1_MODEL_URI = None

with mlflow.start_run(run_name="topic1-lab1-strands-explainer") as run:
    info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=StrandsFraudExplainer(),
        input_example=example_df,
    )
    LAB1_MODEL_URI = info.model_uri
    print("Model URI:", LAB1_MODEL_URI)

# Reload and call. This should print 3 explanation rows.
if LAB1_MODEL_URI is not None:
    loaded = mlflow.pyfunc.load_model(LAB1_MODEL_URI)
    result = loaded.predict(example_df)
    print(result)


## Topic 2: Nested runs and `search_runs`

Hyperparameter sweeps create many runs. Without structure, they clutter the experiment UI and you cannot tell which child run belongs to which sweep, let alone which sweep belongs to which feature branch.

MLflow gives you two tools for this:

### Nested runs

```python
with mlflow.start_run(run_name="parent-sweep") as parent:
    for hp in grid:
        with mlflow.start_run(run_name=f"child-lr{hp['lr']}", nested=True) as child:
            ...  # log params, train, log metrics
```

The MLflow UI shows children indented under the parent. The managed SageMaker MLflow tracking server supports this identically to OSS MLflow - no special handling needed.

Source: `https://mlflow.org/docs/latest/ml/traditional-ml/tutorials/hyperparameter-tuning/notebooks/parent-child-runs/`

### `mlflow.search_runs`

```python
runs_df = mlflow.search_runs(
    experiment_ids=[EXP_ID],
    filter_string='metrics.f1 > 0.8 and params.lr = "0.001"',
    order_by=["metrics.f1 DESC"],
    max_results=50,
)
```

Returns a pandas DataFrame of all matching runs in the experiment, filterable by metric, param, or tag, sortable by any column. Use this to pick the winner of a sweep programmatically instead of clicking around the UI.

**Verified filter syntax** (simplified SQL WHERE):

- AND only - no OR clauses.
- Quoted string literals for tag/param values: `tags.sweep_kind = "lr_demo"`.
- Numeric comparisons unquoted: `metrics.f1 > 0.8`.
- `order_by` is a list of strings with optional `ASC`/`DESC`: `["metrics.f1 DESC"]`.

Source: `https://mlflow.org/docs/latest/ml/search/search-runs/`


In [ ]:
# DEMO: parent + 4 child runs simulating a learning-rate sweep.

import random


# Fake "training" function: a quadratic with the optimum at lr=0.001.
# Lower distance from the optimum lr -> higher f1.
def fake_train(lr, batch_size):
    distance = abs((lr - 0.001) / 0.001)
    base_f1 = 0.92 - 0.15 * distance + random.uniform(-0.01, 0.01)
    return {"f1": max(0.0, base_f1), "loss": 1 - base_f1}


sweep_grid = [
    {"lr": 0.01,   "batch_size": 32},
    {"lr": 0.001,  "batch_size": 32},
    {"lr": 0.0001, "batch_size": 32},
    {"lr": 0.001,  "batch_size": 64},
]

random.seed(42)

with mlflow.start_run(run_name="topic2-demo-lr-sweep") as parent:
    mlflow.set_tag("sweep_kind", "lr_demo")
    mlflow.log_param("grid_size", len(sweep_grid))

    for i, hp in enumerate(sweep_grid):
        with mlflow.start_run(run_name=f"child-{i}-lr{hp['lr']}", nested=True) as child:
            mlflow.log_params(hp)
            metrics = fake_train(**hp)
            mlflow.log_metrics(metrics)
            mlflow.set_tag("child_index", str(i))
            print(f"  child {i}: lr={hp['lr']:.4f}  f1={metrics['f1']:.4f}")

    print(f"\nParent run id: {parent.info.run_id}")


In [ ]:
# DEMO: query the sweep with search_runs and pick the winner programmatically.

runs_df = mlflow.search_runs(
    experiment_ids=[EXP_ID],
    filter_string='tags.sweep_kind = "lr_demo"',
    order_by=["metrics.f1 DESC"],
    max_results=50,
)

# The parent has no metrics.f1, so it ends up at the bottom. Drop it.
sweep_children = runs_df[runs_df["metrics.f1"].notna()].copy()
print(f"Found {len(sweep_children)} child runs in sweep_kind=lr_demo\n")
print(sweep_children[["run_id", "tags.mlflow.runName", "params.lr", "params.batch_size", "metrics.f1"]])

best = sweep_children.iloc[0]
print(f"\nBest run: {best['tags.mlflow.runName']} with f1={best['metrics.f1']:.4f}")
print(f"Best run_id: {best['run_id']}")


### Lab 2: Run your own threshold sweep and pick the winner

In Lab 1 you wrapped a fraud explainer agent. But the threshold pyfunc from the Topic 1 demo (`ThresholdAlertModel`) has two hyperparameters: `score_threshold` and `amount_cap_usd`. You want to find the combination that maximizes a "captured fraud value" metric on a held-out batch, without blocking too many legitimate transactions.

**Steps**:

1. Use the provided `BATCH_DF` (a synthetic batch of 100 transactions with `fraud_score`, `amount_usd`, and `is_fraud` ground truth).
2. Run a sweep over a 3-by-3 grid of `score_threshold` in `[0.5, 0.7, 0.9]` and `amount_cap_usd` in `[1000, 3000, 5000]`. That gives 9 child runs.
3. For each cell of the grid, inside a nested run, log:
   - **params**: `score_threshold`, `amount_cap_usd`.
   - **metrics**: `captured_fraud_usd` (sum of `amount_usd` over rows where the model said `block` AND `is_fraud == 1`), `false_block_usd` (sum where the model said `block` AND `is_fraud == 0`).
   - **tag**: `sweep_kind = "threshold_lab"`.
4. After the sweep, use `mlflow.search_runs()` to find ALL children of this sweep. Rank them by `captured_fraud_usd / (1 + false_block_usd)` (so a model that captures fraud but also blocks a lot of innocent customers ranks lower).
5. Print the winning params.

**Success criterion**: the winning print contains both `score_threshold=` and `amount_cap_usd=` values, and the DataFrame from `search_runs` has exactly 9 child rows tagged `threshold_lab`.


In [ ]:
# SOLUTION: Lab 2 - Threshold sweep with nested runs + search_runs winner pick

import numpy as np

np.random.seed(0)
N = 100
BATCH_DF = pd.DataFrame({
    "fraud_score": np.clip(np.random.beta(2, 5, N) + np.random.normal(0, 0.1, N), 0, 1),
    "amount_usd":  np.random.gamma(2.0, 800, N).round(2),
})
BATCH_DF["is_fraud"] = (
    (BATCH_DF["fraud_score"] > 0.6) & (BATCH_DF["amount_usd"] > 500)
).astype(int)

print(f"Batch size: {len(BATCH_DF)}, fraud rate: {BATCH_DF['is_fraud'].mean():.2%}")

threshold_grid = [0.5, 0.7, 0.9]
cap_grid       = [1000, 3000, 5000]

LAB2_PARENT_RUN_ID = None

with mlflow.start_run(run_name="topic2-lab2-threshold-sweep") as parent:
    mlflow.set_tag("sweep_kind", "threshold_lab")
    LAB2_PARENT_RUN_ID = parent.info.run_id

    for thr in threshold_grid:
        for cap in cap_grid:
            with mlflow.start_run(run_name=f"thr{thr}-cap{cap}", nested=True):
                # 1) Build a ThresholdAlertModel with these hyperparameters.
                model = ThresholdAlertModel(score_threshold=thr, amount_cap_usd=cap)
                # 2) Score BATCH_DF (drop is_fraud first when calling predict).
                preds = model.predict(None, BATCH_DF[["fraud_score", "amount_usd"]])
                blocked = preds["action"] == "block"
                # 3) Compute captured_fraud_usd and false_block_usd.
                captured = float(BATCH_DF.loc[blocked & (BATCH_DF["is_fraud"] == 1), "amount_usd"].sum())
                false_block = float(BATCH_DF.loc[blocked & (BATCH_DF["is_fraud"] == 0), "amount_usd"].sum())
                # 4) Log params, metrics, and the sweep_kind tag.
                mlflow.log_params({"score_threshold": thr, "amount_cap_usd": cap})
                mlflow.log_metrics({"captured_fraud_usd": captured, "false_block_usd": false_block})
                mlflow.set_tag("sweep_kind", "threshold_lab")

# Use mlflow.search_runs to find the winner.
runs_df = mlflow.search_runs(
    experiment_ids=[EXP_ID],
    filter_string='tags.sweep_kind = "threshold_lab"',
    order_by=["metrics.captured_fraud_usd DESC"],
)
# Drop parent rows (no metric value)
children = runs_df[runs_df["metrics.captured_fraud_usd"].notna()].copy()
# Rank by captured / (1 + false_block) so heavy false-blockers fall.
children["score"] = children["metrics.captured_fraud_usd"] / (1 + children["metrics.false_block_usd"])
children = children.sort_values("score", ascending=False)
winner = children.iloc[0]
print(f"Found {len(children)} child runs.")
print(f"Winner: score_threshold={winner['params.score_threshold']} amount_cap_usd={winner['params.amount_cap_usd']}")
print(f"  captured_fraud_usd={winner['metrics.captured_fraud_usd']:.2f}  false_block_usd={winner['metrics.false_block_usd']:.2f}")


## Topic 3: Model lineage

The question that always comes up in production: "where did this deployed model come from?" To answer it you need to link, in both directions:

```
registered Model Package  <->  training run  <->  data version  <->  source code commit
```

### What SageMaker gives you out of the box

`sagemaker_client.describe_model_package(ModelPackageName=arn)` returns the package metadata. The **training job ARN is NOT a top-level field**. You have two ways to recover it:

- **Easy**: when you call `create_model_package`, pass it in `CustomerMetadataProperties` (a string-to-string dict) as `training_job_arn`. Then `describe_model_package()` returns it under the same key. Same trick for `git_commit`, `data_version`, `mlflow_run_id`, anything else you want.
- **Advanced**: use SageMaker ML Lineage Tracking. Call `sagemaker_client.list_associations(DestinationArn=model_package_arn)` and follow the source ARNs back to the training job. SageMaker builds this graph automatically when you use Training Jobs and Pipelines.

We will demo the easy option. The homework section points at the advanced option.

Source: `https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/sagemaker/client/describe_model_package.html`

### Bidirectional lineage with MLflow

For MLflow runs the lineage works both ways:

- The Model Package gets a `mlflow_run_id` entry in `CustomerMetadataProperties`.
- The MLflow run gets a `model_package_arn` tag via `mlflow.set_tag()`.

From either end, ONE lookup answers "where did this come from?" That is the property you want in production.


In [ ]:
# DEMO: register a model with full lineage metadata, then round-trip it.
# Note: in the real main notebook the ModelDataUrl would point at the actual
# training-job output tarball. We reuse the threshold pyfunc here just to
# illustrate the registry call. The lineage TAGS are what matter for this topic.

# Find a recent training job to attach lineage to (would normally be your
# main-notebook training_job_name).
recent = sagemaker_client.list_training_jobs(
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=5,
)["TrainingJobSummaries"]

if not recent:
    print("No training jobs found. Run the main Week 19 notebook first.")
    raise RuntimeError("No training job in this account")

MAIN_TRAINING_JOB_NAME = recent[0]["TrainingJobName"]
MAIN_TRAINING_JOB_ARN  = recent[0]["TrainingJobArn"]
print(f"Using training job: {MAIN_TRAINING_JOB_NAME}")

# Pretend git commit + data version (in real life: from os.environ + Delta history)
GIT_COMMIT   = "abc123def"
DATA_VERSION = "42"

with mlflow.start_run(run_name="topic3-demo-lineage") as run:
    mlflow.set_tag("git_commit",       GIT_COMMIT)
    mlflow.set_tag("data_version",     DATA_VERSION)
    mlflow.set_tag("training_job_arn", MAIN_TRAINING_JOB_ARN)

    # Log the threshold pyfunc so we have something concrete to register
    info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=ThresholdAlertModel(),
        input_example=BATCH_DF[["fraud_score", "amount_usd"]].head(3),
    )

    pkg_group = f"week19-optional-demo-{USER_ID}"
    try:
        sagemaker_client.create_model_package_group(ModelPackageGroupName=pkg_group)
    except sagemaker_client.exceptions.ResourceInUse:
        pass

    pkg_response = sagemaker_client.create_model_package(
        ModelPackageGroupName=pkg_group,
        ModelPackageDescription=f"Demo threshold model, MLflow run {run.info.run_id}",
        InferenceSpecification={
            "Containers": [{
                "Image": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-inference:2.1.0-transformers4.36.0-cpu-py310-ubuntu20.04",
                "ModelDataUrl": info.model_uri.replace("runs:/", f"s3://{S3_BUCKET}/mlflow/"),
            }],
            "SupportedContentTypes":      ["application/json"],
            "SupportedResponseMIMETypes": ["application/json"],
        },
        CustomerMetadataProperties={
            "mlflow_run_id":    run.info.run_id,
            "training_job_arn": MAIN_TRAINING_JOB_ARN,
            "git_commit":       GIT_COMMIT,
            "data_version":     DATA_VERSION,
        },
        ModelApprovalStatus="PendingManualApproval",
    )
    DEMO_PACKAGE_ARN = pkg_response["ModelPackageArn"]

    # Bidirectional lineage: tag the MLflow run with the package ARN
    mlflow.set_tag("model_package_arn", DEMO_PACKAGE_ARN)
    print(f"Registered: {DEMO_PACKAGE_ARN}")

# Round-trip: pull lineage back out of the registry
detail = sagemaker_client.describe_model_package(ModelPackageName=DEMO_PACKAGE_ARN)
print("\nLineage metadata recovered from registry:")
for k, v in detail.get("CustomerMetadataProperties", {}).items():
    print(f"  {k:20s} {v}")


### Lab 3: Build a lineage lookup function

Write a function `trace_model(package_arn: str) -> dict` that, given a Model Package ARN, returns a dict with these keys:

- `package_arn`: the input ARN.
- `mlflow_run_id`: from `CustomerMetadataProperties`.
- `training_job_arn`: from `CustomerMetadataProperties`.
- `git_commit`: from `CustomerMetadataProperties`.
- `data_version`: from `CustomerMetadataProperties`.
- `training_job_status`: by calling `sagemaker_client.describe_training_job(TrainingJobName=...)`. You will need to parse the training-job NAME from the training-job ARN (it is the last `/`-separated segment).
- `mlflow_run_tags`: a dict of tag name to value, by calling `mlflow.get_run(mlflow_run_id).data.tags`.

Then call `trace_model(DEMO_PACKAGE_ARN)` and pretty-print the result.

**Success criterion**: every key has a non-None value, and the printed dict shows the same `git_commit` you set in Cell 17 (`abc123def`).


In [ ]:
# SOLUTION: Lab 3 - Build a lineage lookup function

def trace_model(package_arn: str) -> dict:
    # 1) describe_model_package -> CustomerMetadataProperties
    detail = sagemaker_client.describe_model_package(ModelPackageName=package_arn)
    props = detail.get("CustomerMetadataProperties", {}) or {}

    mlflow_run_id    = props.get("mlflow_run_id")
    training_job_arn = props.get("training_job_arn")
    git_commit       = props.get("git_commit")
    data_version     = props.get("data_version")

    # 2) Parse training job NAME from the ARN (last "/"-separated segment).
    training_job_status = None
    if training_job_arn:
        training_job_name = training_job_arn.split("/")[-1]
        # 3) describe_training_job -> TrainingJobStatus
        try:
            tj = sagemaker_client.describe_training_job(TrainingJobName=training_job_name)
            training_job_status = tj.get("TrainingJobStatus")
        except Exception as e:
            training_job_status = f"LOOKUP_FAILED: {e}"

    # 4) mlflow.get_run -> data.tags
    mlflow_run_tags = {}
    if mlflow_run_id:
        try:
            mlflow_run_tags = dict(mlflow.get_run(mlflow_run_id).data.tags)
        except Exception as e:
            mlflow_run_tags = {"LOOKUP_FAILED": str(e)}

    return {
        "package_arn":         package_arn,
        "mlflow_run_id":       mlflow_run_id,
        "training_job_arn":    training_job_arn,
        "git_commit":          git_commit,
        "data_version":        data_version,
        "training_job_status": training_job_status,
        "mlflow_run_tags":     mlflow_run_tags,
    }


result = trace_model(DEMO_PACKAGE_ARN)
print(json.dumps(result, indent=2, default=str))


## Think About It (cross-topic) and Homework Extensions

### Reflection (no group activity, just self-reflection)

1. The `ThresholdAlertModel` pyfunc in Topic 1 is stateless: the params live on the instance. The `StrandsFraudExplainer` pyfunc in Lab 1 has live state (the agent built in `load_context`). Which one is safer to serve in a multi-replica endpoint, and why? What kinds of bugs could the stateful one hide that the stateless one cannot?

2. In Topic 2 you used `tags.sweep_kind = "threshold_lab"` to mark the sweep. If you ran ten different sweeps in the same experiment, how would your `search_runs` filter need to change to find ONLY the latest sweep? Hint: tags can carry a timestamp.

3. In Topic 3 you stored `training_job_arn` in `CustomerMetadataProperties`. What would happen to the lineage if someone deleted the training job after 90 days but kept the model package? Which of your `trace_model` fields would still work? Which would break? What is the operational lesson?

### Homework extensions (async, no rubric)

- **Topic 1**: serve the threshold pyfunc behind a SageMaker Serverless endpoint using `mlflow.sagemaker.deploy()`. Try it with the same `BATCH_DF`. Compare the latency to running it locally.
- **Topic 2**: convert the threshold sweep from a nested loop to use Optuna with the MLflow Optuna callback. Compare the resulting run tree in the MLflow UI.
- **Topic 3**: replace the `CustomerMetadataProperties` approach with SageMaker ML Lineage Tracking. Call `sagemaker_client.list_associations(DestinationArn=DEMO_PACKAGE_ARN)` and reconstruct the chain `training_job -> model -> model_package -> endpoint`. Compare which approach is more discoverable when you have 100 model packages.
